# 01 — Pokémon dataset EDA (Phase 0)

Three concrete decisions to come out of this EDA, that drive Phase 1:

1. Class-balancing strategy (depending on how unbalanced `type1` is)
2. Image preprocessing (resize, alpha handling)
3. Normalization stats

Sections:
1. Setup
2. Load and inspect metadata
3. Type distribution
4. Single-image inspection
5. Pixel statistics
6. Wrap-up

---
## 1. Setup

In [9]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from PIL import Image

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  
DATA_DIR = PROJECT_ROOT / "data"
IMAGES_DIR = DATA_DIR / "images"
METADATA_PATH = DATA_DIR / "metadata.csv"

%matplotlib inline

---
## 2. Load and inspect metadata


In [ ]:
df = pd.read_csv(METADATA_PATH)
print(df.head())
print("\nShape :", df.shape)
print("\nMissing values :\n", df.isna().sum())
print("\nDuplicate ids :", df.duplicated(subset='id').sum())
print("\nDtypes :\n", df.dtypes)

In [11]:
ids_on_disk = sorted(int(f.stem) for f in IMAGES_DIR.glob("*.png"))
print((df['id'] == ids_on_disk).mean()) 

1.0


Every pokemon in the csv has its image in the images folder. CSV is clean.

---
## 3. Type distribution

**Goal.** Understand class imbalance — this is the **single most important EDA insight** for our project, because `type1` is the conditioning signal of the diffusion model.

**Questions to answer:**
- How many unique types are there? (Pokémon canonically has 18.)
- What's the count of the most-common type? Of the least-common?
- What's the imbalance ratio (max / min)?
- Are there any types with fewer than ~30 examples? (Below that, generation per class will likely be poor.)

**Hints:**
- `df['type1'].nunique()`, `df['type1'].value_counts()`
- `matplotlib.pyplot.bar` on the sorted value counts
- Sort the bar chart descending — easier to read.

**Decision this step informs:**
- If imbalance ratio < 3 → no special handling, regular shuffled batches are fine.
- If 3 ≤ ratio < 10 → consider using `WeightedRandomSampler` in the DataLoader (Phase 1).
- If ratio ≥ 10 → some classes may need oversampling or augmentation.

In [ ]:
type_counts = df["type1"].value_counts()
type_counts.plot(kind='bar')
plt.title("Number of pokemon by main type")
plt.xlabel("Main type")
plt.ylabel("Count")
plt.show()

In [ ]:
imbalance_ratio = type_counts.max() / type_counts.min()
print(f"Imbalance ratio : {imbalance_ratio:.2f}")
print(f"Most common  : {type_counts.idxmax()} ({type_counts.max()})")
print(f"Least common : {type_counts.idxmin()} ({type_counts.min()})")

Around 15× more water than flying pokemon as `type1`. Way too unbalanced to ignore — a weighted sampler in the dataloader will be needed in Phase 1.

---
## 4. Single-image inspection

**Goal.** Understand what one sample looks like at the pixel level — this drives every preprocessing decision.

**Questions to answer:**
- What is the native size of a sprite? (Width × Height in pixels)
- What is the image mode (`'RGB'`, `'RGBA'`, `'L'`)?
- Is there an alpha channel? What does it encode?
- What does the background look like? (Transparent vs. white vs. black)
- What are the pixel value ranges? `[0, 255]` (uint8) or `[0, 1]` (float)?
- If there's an alpha channel, what's its content? Mostly 0 (transparent) and 255 (opaque), or smooth gradients?

**Hints:**
- Open Bulbasaur (id=1) with `PIL.Image.open(IMAGES_DIR / '0001.png')`.
- `img.size`, `img.mode`, `np.array(img).shape`, `np.array(img).dtype`
- Display with `plt.imshow(np.array(img))`. If RGBA, also display each channel separately to inspect the alpha.

**Decision this step informs:**
- If sprites are RGBA → we'll need to **handle the alpha channel** in Phase 1. Common strategies: (a) flatten onto a fixed-color background (e.g., white), (b) use it as a 4th input channel, (c) use it as a mask. Most diffusion baselines do (a).
- The native size determines the resize policy (we plan 64×64 for training).

In [ ]:
img_0001 = Image.open(IMAGES_DIR / "0001.png").convert("RGBA")
print("Size :", img_0001.size)
print("Mode :", img_0001.mode)

img_0001

In [ ]:
arr_img = np.array(img_0001)
print(f"Array shape : {arr_img.shape}")
print(f"Dtype       : {arr_img.dtype}")
print(f"Pixel range : [{arr_img.min()}, {arr_img.max()}]")

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(arr_img)
axes[0].set_title("RGBA")
axes[0].axis("off")
axes[1].imshow(arr_img[:, :, 3], cmap="gray")
axes[1].set_title("Alpha")
axes[1].axis("off")
plt.show()

---
## 5. Pixel statistics

**Goal.** Compute the per-channel **mean** and **std** of the dataset — these go into the normalization step of the diffusion data pipeline.

**Questions to answer:**
- What are the per-channel means (R, G, B) in `[0, 1]` after dividing by 255?
- What are the per-channel stds?
- If sprites are RGBA, **how should we treat the alpha** before computing stats? (Composite onto white? Drop alpha? Mask out transparent pixels?)
- Do the global stats look reasonable, or are they dominated by transparent / background pixels?

**Hints:**
- Iterate over all images, accumulate sum and sum-of-squares per channel.
- Or: stack a sample of N images into a `(N, H, W, C)` array and use `np.mean / np.std` on axes `(0, 1, 2)`.
- Decide *now* what to do with alpha — the stats on raw RGBA vs. RGB-on-white are very different.

**Decision this step informs:**
- The `transforms.Normalize(mean, std)` call inside the future `PokemonDataset` (Phase 0 step 3 → `src/data.py`).
- Alternatively: many diffusion baselines use the simpler `[-1, 1]` rescaling (`x = 2*x - 1`), which is independent of the dataset stats. We can choose which one in Phase 1.

Compositing on white → output is plain RGB in `[0, 1]`. Same thing the dataset will hand to the model later, so the stats below are representative.

One sprite (#910 Sandy Shocks) is `96x95` instead of `96x96`, resized with bilinear before stacking.

In [ ]:
TARGET_SIZE = (96, 96)


def composite_on_white(img_rgba):
    """RGBA → RGB on white, float32 in [0, 1]."""
    arr = np.array(img_rgba, dtype=np.float32) / 255.0
    rgb, alpha = arr[..., :3], arr[..., 3:4]
    white = np.ones_like(rgb)
    return rgb * alpha + white * (1 - alpha)


all_images = []
for f in sorted(IMAGES_DIR.glob("*.png")):
    img = Image.open(f).convert("RGBA")
    if img.size != TARGET_SIZE:
        img = img.resize(TARGET_SIZE, Image.BILINEAR)  # only #910 hits this branch
    all_images.append(composite_on_white(img))

stack = np.stack(all_images)   # (N, H, W, 3)
mean = stack.mean(axis=(0, 1, 2))
std = stack.std(axis=(0, 1, 2))

print(f"Stack shape : {stack.shape}")
print(f"Mean (R, G, B) : ({mean[0]:.4f}, {mean[1]:.4f}, {mean[2]:.4f})")
print(f"Std  (R, G, B) : ({std[0]:.4f}, {std[1]:.4f}, {std[2]:.4f})")

---
## 6. Wrap-up

Pipeline planned for `src/data.py`:
- keep native 96×96 (only #910 Sandy Shocks needs a bilinear resize from 96×95)
- composite RGBA on white, output RGB float32 in `[0, 1]`
- normalize to `[-1, 1]` (`x = 2 * x - 1`)
- weighted sampler with `1 / count(type1)` to fight the 14.9× imbalance

No records dropped — all 1025 are usable. Two PNGs (#678 Meowstic, #696 Tyrunt) had a corrupted `iCCP` chunk that PIL refused; stripped it manually so the actual pixel data is preserved.

To revisit in Phase 1:
- maybe keep alpha as a 4th channel and let the model learn the silhouette
- check if the white background biases unconditional samples toward white — if so, switch to noise

Numbers to remember:
- 1025 samples / 18 types
- water 134 / flying 9 → ratio 14.89
- mean (R, G, B) = (0.91, 0.90, 0.90)
- std  (R, G, B) = (0.23, 0.24, 0.25)